In [1]:
import os

import numpy as np
import pandas as pd
from fcd import calculate_frechet_distance, canonical_smiles, get_fcd, get_predictions, load_ref_model
from rdkit import RDLogger
from rdkit.Chem.rdmolfiles import MolFromSmiles, MolToSmiles
from rich.progress import track

In [2]:
RDLogger.DisableLog("rdApp.*")

np.random.seed(0)
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # set gpu

In [3]:
# Load chemnet model
model = load_ref_model()

/data/dragon320/buttensc/Storage/Applications/miniforge/envs/evaluation/lib/python3.11/site-packages/fcd/fcd.py:42: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model_confi

In [4]:
def canonical(smi):
    try:
        return MolToSmiles(MolFromSmiles(smi))  # type: ignore
    except Exception:
        return None


In [5]:
# follow example
# https://github.com/bioinf-jku/FCD/blob/master/example.ipynb

In [6]:
# Load molecules
mols = {
    "geom_drugs": "data/03_evaluation/geom_drugs.csv",
    "drugbank_2d": "data/03_evaluation/drugbank_approved_2d.csv",
    "drugbank_3d": "data/03_evaluation/drugbank_approved_3d.csv",
    "eqgat": "data/03_evaluation/eqgat_100000_predictions.csv",
    "gcdm": "data/03_evaluation/gcdm_100000_predictions.csv",
    "flowmol": "data/03_evaluation/flowmol_100000_predictions.csv",
    "geoldm": "data/03_evaluation/geoldm_100000_predictions.csv",
    "semlaflow": "data/03_evaluation/semlaflow_100000_predictions.csv",
}

smiles = {k: pd.read_csv(v, low_memory=False)["smiles"].astype("string").dropna().values for k, v in track(mols.items())}
# smiles = {k: [canonical(s) for s in track(smiles[k])] for k, v in mols.items()}

Output()

In [7]:
# activations = {k: get_predictions(model, smiles[k]) for k in track(smiles.keys())}
# mus = {k: np.mean(activations[k], axis=0) for k in track(smiles.keys())}
# covs = {k: np.cov(activations[k]) for k in track(smiles.keys())}


In [14]:
# IMPORTANT: take at least 10000 molecules as FCD can vary with sample size
sample1 = np.random.choice(smiles["geom_drugs"], 10000, replace=False)
# sample2 = smiles["drugbank_3d"]
sample2 = smiles["eqgat"]
sample2 = smiles["gcdm"]

# get CHEBMLNET activations of generated molecules
fcd_score = get_fcd(sample1, sample2, model)
print("FCD: ", round(fcd_score, 2))

FCD:  41.81


In [9]:
# get CHEBMLNET activations of generated molecules
act1 = get_predictions(model, sample1)
act2 = get_predictions(model, sample2)

mu1 = np.mean(act1, axis=0)
sigma1 = np.cov(act1.T)

mu2 = np.mean(act2, axis=0)
sigma2 = np.cov(act2.T)

fcd_score = calculate_frechet_distance(mu1=mu1, mu2=mu2, sigma1=sigma1, sigma2=sigma2)

print("FCD: ", fcd_score)

FCD:  5.566871221149924


In [13]:
mu1.shape

(512,)

In [36]:
# data = np.fromfile("data/03_evaluation/semlaflow_100000_predictions.fcd", dtype=np.float64)
data = np.load("data/03_evaluation/semlaflow_100000_predictions.fcd.npy", allow_pickle=True)
data.shape


(94926, 512)

In [38]:
data.shape

(94926, 512)